# Notebook 07 — Frozen Full-Train XGBoost Training

Fit the Stage 06 primary XGBoost once on the complete eligible train-only
candidate population using full-train, within-symbol Average Uniqueness.

No unseen-test access, probability calibration, threshold selection, or
in-sample model-quality reporting occurs in this notebook.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import time

import joblib
import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("joblib:", joblib.__version__)


## 1. Repository and project imports

In [ ]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / "configs").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "src").exists()
        ):
            return candidate
    raise FileNotFoundError("Repository root was not found.")

REPOSITORY_ROOT = locate_repository_root(Path.cwd())
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.models.frozen_training import (
    FROZEN_TRAINING_SCHEMA_VERSION,
    dataframe_fingerprint,
    fitted_feature_names,
    save_pipeline_with_reload_check,
    weighted_scale_pos_weight,
)
from src.models.tuning import feature_columns_from_manifest
from src.models.xgboost_model import build_xgboost_pipeline
from src.utils.paths import data_paths, repository_result_paths, resolve_data_root
from src.utils.reproducibility import git_commit_sha, software_manifest, stable_object_hash
from src.validation.purged_walk_forward import normalize_candidate_event_panel
from src.validation.sample_weighting import (
    compute_full_train_average_uniqueness,
    summarize_average_uniqueness_population,
)

print("Repository root:", REPOSITORY_ROOT)


## 2. Frozen configurations and lineage

In [ ]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")
    return value


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = json.load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"JSON artifact must be an object: {path}")
    return value

paths_config = load_yaml(REPOSITORY_ROOT / "configs" / "paths.yaml")
frozen_training_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "frozen_training.yaml"
)
DATA_ROOT = resolve_data_root(paths_config, REPOSITORY_ROOT)
DATA_PATHS = data_paths(DATA_ROOT, paths_config)
RESULT_PATHS = repository_result_paths(REPOSITORY_ROOT, paths_config)
CANDIDATE_TRAIN_PATH = DATA_PATHS["candidate_data"] / "train"
LABELED_TRAIN_PATH = DATA_PATHS["labeled_data"] / "train"

feature_manifest_path = RESULT_PATHS["manifests"] / "04_approved_model_features.csv"
stage04_policy_path = RESULT_PATHS["manifests"] / "04_candidate_filter_policy.json"
stage05_manifest_path = RESULT_PATHS["manifests"] / "05_purged_anchored_walk_forward_manifest.json"
stage06_decision_path = RESULT_PATHS["manifests"] / "06_model_selection_decision.json"
stage06_manifest_path = RESULT_PATHS["manifests"] / "06_optuna_model_selection_manifest.json"
best_hyperparameters_path = RESULT_PATHS["optuna"] / "06_best_hyperparameters.json"

for path in [
    feature_manifest_path,
    stage04_policy_path,
    stage05_manifest_path,
    stage06_decision_path,
    stage06_manifest_path,
    best_hyperparameters_path,
]:
    if not path.exists():
        raise FileNotFoundError(f"Required frozen artifact is missing: {path}")

feature_manifest_df = pd.read_csv(feature_manifest_path, low_memory=False)
stage04_policy = load_json(stage04_policy_path)
stage05_manifest = load_json(stage05_manifest_path)
stage06_decision = load_json(stage06_decision_path)
stage06_manifest = load_json(stage06_manifest_path)
best_hyperparameters = load_json(best_hyperparameters_path)
MODEL_FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES = (
    feature_columns_from_manifest(feature_manifest_df)
)

SEED = int(frozen_training_config["model_fit"]["random_seed"])
SELECTED_MODEL = str(stage06_decision["primary_selected_model"])
SELECTED_PARAMS = dict(best_hyperparameters[SELECTED_MODEL]["params"])
SELECTED_TRIAL_NUMBER = int(
    best_hyperparameters[SELECTED_MODEL]["selected_trial_number"]
)

assert frozen_training_config["status"] == "stage_07_configured_v1"
assert FROZEN_TRAINING_SCHEMA_VERSION == "stage07_v1_full_train_average_uniqueness_xgboost"
assert SELECTED_MODEL == "xgboost"
assert SELECTED_TRIAL_NUMBER == 25
assert stage06_decision["challenger_model"] == "random_forest"
assert stage06_decision["threshold_selected"] is False
assert stage06_decision["unseen_test_used"] is False
assert stage06_manifest["status"] == "frozen_after_integrity_checks"
assert stage06_manifest["unseen_test_used"] is False
assert stage05_manifest["unseen_test_used"] is False
assert stage04_policy["primary_side"] == "long"
assert np.isclose(float(stage04_policy["primary_threshold_fraction"]), 0.15)
assert len(MODEL_FEATURES) == 35
assert len(NUMERIC_FEATURES) == 34
assert CATEGORICAL_FEATURES == ["gmma_state"]
assert int(SELECTED_PARAMS["n_estimators"]) == 150
assert int(SELECTED_PARAMS["max_depth"]) == 2
assert SELECTED_PARAMS["class_weight_mode"] == "none"

lineage_audit_df = pd.DataFrame([{
    "stage05_configuration_hash": stage05_manifest["configuration_hash"],
    "stage06_code_commit_sha": stage06_manifest["git_commit_sha"],
    "stage06_study_signature": stage06_manifest["study_signature"],
    "selected_model": SELECTED_MODEL,
    "selected_trial_number": SELECTED_TRIAL_NUMBER,
    "threshold_selected": False,
    "calibration_fitted": False,
    "unseen_test_used": False,
}])
lineage_audit_df.to_csv(
    RESULT_PATHS["audits"] / "07_input_artifact_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(lineage_audit_df)
print("Selected parameters:", SELECTED_PARAMS)


## 3. Complete Stage 04 train candidate population

In [ ]:
candidate_files = sorted(CANDIDATE_TRAIN_PATH.glob("*_train_candidates.csv"))
EXPECTED_EVENTS = int(frozen_training_config["training_population"]["expected_events"])
EXPECTED_SYMBOLS = int(frozen_training_config["training_population"]["expected_symbols"])


def candidate_symbol_from_path(path: Path) -> str:
    suffix = "_train_candidates.csv"
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected candidate file: {path.name}")
    return path.name[:-len(suffix)]

candidate_parts = []
candidate_error_rows = []
usecols = ["dEven", "event_end_date", "meta_label", *MODEL_FEATURES]
started = time.perf_counter()

for file_number, path in enumerate(candidate_files, start=1):
    symbol = candidate_symbol_from_path(path)
    try:
        frame = pd.read_csv(path, usecols=usecols, low_memory=False)
        frame.insert(0, "symbol", symbol)
        event_dates = pd.to_datetime(frame["dEven"], errors="coerce")
        frame.insert(0, "event_id", symbol + "::" + event_dates.dt.strftime("%Y-%m-%d"))
        candidate_parts.append(frame)
    except Exception as exc:
        candidate_error_rows.append({
            "symbol": symbol,
            "file_name": path.name,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })
    if file_number == 1 or file_number % 50 == 0 or file_number == len(candidate_files):
        print(
            f"[candidate panel] [{file_number:>4}/{len(candidate_files)}] "
            f"elapsed={time.perf_counter()-started:,.1f}s errors={len(candidate_error_rows)}"
        )

candidate_errors_df = pd.DataFrame(
    candidate_error_rows,
    columns=["symbol", "file_name", "error_type", "error_message"],
)
candidate_errors_df.to_csv(
    RESULT_PATHS["audits"] / "07_training_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
if not candidate_errors_df.empty:
    raise RuntimeError("Candidate loading errors exist. Review 07_training_errors.csv")

candidate_panel = normalize_candidate_event_panel(pd.concat(candidate_parts, ignore_index=True))
for feature in NUMERIC_FEATURES:
    candidate_panel[feature] = pd.to_numeric(candidate_panel[feature], errors="coerce")
candidate_panel["gmma_state"] = candidate_panel["gmma_state"].astype("string")
candidate_panel = candidate_panel.sort_values(
    ["symbol", "dEven", "event_id"], kind="stable"
).reset_index(drop=True)

numeric_array = candidate_panel[NUMERIC_FEATURES].to_numpy(dtype=float)
assert len(candidate_panel) == EXPECTED_EVENTS
assert candidate_panel["symbol"].nunique() == EXPECTED_SYMBOLS
assert candidate_panel["event_id"].is_unique
assert candidate_panel["meta_label"].isin([0, 1]).all()
assert not np.isinf(numeric_array).any()

fingerprint_columns = [
    "event_id", "symbol", "dEven", "event_end_date", "meta_label", *MODEL_FEATURES
]
training_population_fingerprint = dataframe_fingerprint(candidate_panel, fingerprint_columns)
training_population_audit_df = pd.DataFrame([{
    "candidate_events": int(len(candidate_panel)),
    "symbols": int(candidate_panel["symbol"].nunique()),
    "first_event_start_date": candidate_panel["dEven"].min(),
    "last_event_start_date": candidate_panel["dEven"].max(),
    "last_event_end_date": candidate_panel["event_end_date"].max(),
    "positive_events": int(candidate_panel["meta_label"].sum()),
    "negative_events": int((candidate_panel["meta_label"] == 0).sum()),
    "positive_fraction": float(candidate_panel["meta_label"].mean()),
    "features": int(len(MODEL_FEATURES)),
    "numeric_missing_values": int(candidate_panel[NUMERIC_FEATURES].isna().sum().sum()),
    "categorical_missing_values": int(candidate_panel[CATEGORICAL_FEATURES].isna().sum().sum()),
    "infinite_numeric_values": int(np.isinf(numeric_array).sum()),
    "duplicate_event_ids": int(candidate_panel["event_id"].duplicated().sum()),
    "training_population_sha256": training_population_fingerprint,
    "unseen_test_used": False,
}])
training_population_audit_df.to_csv(
    RESULT_PATHS["audits"] / "07_training_population_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(training_population_audit_df)


## 4. Symbol train calendars and full-train Average Uniqueness

In [ ]:
def labeled_symbol_from_path(path: Path) -> str:
    suffix = "_train_labeled.csv"
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected labeled-train file: {path.name}")
    return path.name[:-len(suffix)]

symbol_calendars = {}
calendar_error_rows = []
for path in sorted(LABELED_TRAIN_PATH.glob("*_train_labeled.csv")):
    symbol = labeled_symbol_from_path(path)
    try:
        frame = pd.read_csv(path, usecols=["dEven"], low_memory=False)
        calendar = pd.DatetimeIndex(sorted(
            pd.to_datetime(frame["dEven"], errors="coerce")
            .dropna().dt.normalize().unique()
        ))
        if calendar.empty or calendar.has_duplicates:
            raise ValueError("Invalid symbol train calendar.")
        symbol_calendars[symbol] = calendar
    except Exception as exc:
        calendar_error_rows.append({
            "symbol": symbol,
            "file_name": path.name,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })

calendar_errors_df = pd.DataFrame(
    calendar_error_rows,
    columns=["symbol", "file_name", "error_type", "error_message"],
)
calendar_errors_df.to_csv(
    RESULT_PATHS["audits"] / "07_symbol_calendar_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
if not calendar_errors_df.empty:
    raise RuntimeError("Symbol calendar errors exist.")
assert set(symbol_calendars) == set(candidate_panel["symbol"].astype(str))
assert len(symbol_calendars) == EXPECTED_SYMBOLS

weight_input = candidate_panel[["event_id", "symbol", "dEven", "event_end_date"]].copy()
full_train_weights_df, full_train_weight_symbol_audit_df = (
    compute_full_train_average_uniqueness(weight_input, symbol_calendars)
)
weight_by_event_id = full_train_weights_df.set_index("event_id")["sample_weight"]
full_train_sample_weight = (
    candidate_panel["event_id"].astype(str).map(weight_by_event_id).to_numpy(dtype=float)
)
assert len(full_train_weights_df) == len(candidate_panel)
assert full_train_weights_df["event_id"].is_unique
assert np.isfinite(full_train_sample_weight).all()
assert (full_train_sample_weight > 0.0).all()
assert np.isclose(full_train_sample_weight.mean(), 1.0, atol=1e-12, rtol=1e-12)
assert full_train_weights_df["weight_source_scope"].eq(
    "complete_eligible_train_candidate_population"
).all()

weight_summary = summarize_average_uniqueness_population(
    full_train_weights_df,
    population_id="complete_eligible_train_candidate_population",
    validation_events_used=0,
    unseen_test_events_used=0,
)
full_train_weight_audit_df = pd.DataFrame([weight_summary])
weights_fingerprint = dataframe_fingerprint(
    full_train_weights_df.sort_values("event_id", kind="stable").reset_index(drop=True),
    [
        "event_id", "symbol", "event_start_date", "event_end_date",
        "average_uniqueness_raw", "sample_weight"
    ],
)
full_train_weight_audit_df["weights_sha256"] = weights_fingerprint

full_train_weights_df.to_csv(
    RESULT_PATHS["folds"] / "07_full_train_average_uniqueness_weights.csv",
    index=False,
    encoding="utf-8-sig",
)
full_train_weight_audit_df.to_csv(
    RESULT_PATHS["audits"] / "07_full_train_average_uniqueness_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
full_train_weight_symbol_audit_df.to_csv(
    RESULT_PATHS["audits"] / "07_full_train_average_uniqueness_symbol_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(full_train_weight_audit_df)


## 5. Fit and serialize the frozen XGBoost pipeline

In [ ]:
X_train = candidate_panel[MODEL_FEATURES]
y_train = candidate_panel["meta_label"].to_numpy(dtype=int)
class_weight_mode = str(SELECTED_PARAMS.get("class_weight_mode", "none"))
scale_pos_weight = weighted_scale_pos_weight(
    y_train,
    full_train_sample_weight,
    mode=class_weight_mode,
)

frozen_pipeline = build_xgboost_pipeline(
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    params=SELECTED_PARAMS,
    seed=SEED,
    scale_pos_weight=scale_pos_weight,
)
fit_started = time.perf_counter()
frozen_pipeline.fit(
    X_train,
    y_train,
    model__sample_weight=full_train_sample_weight,
)
fit_seconds = time.perf_counter() - fit_started

transformed_feature_names = fitted_feature_names(frozen_pipeline)
fitted_feature_schema_df = pd.DataFrame({
    "transformed_feature_order": np.arange(1, len(transformed_feature_names) + 1),
    "transformed_feature": transformed_feature_names,
})
fitted_feature_schema_df.to_csv(
    RESULT_PATHS["manifests"] / "07_fitted_feature_schema.csv",
    index=False,
    encoding="utf-8-sig",
)

probe_positions = np.linspace(
    0, len(candidate_panel) - 1, num=min(512, len(candidate_panel)), dtype=int
)
X_probe = X_train.iloc[probe_positions]
model_path = RESULT_PATHS["models"] / "07_frozen_xgboost_pipeline.joblib"
serialization_audit = save_pipeline_with_reload_check(
    frozen_pipeline, model_path, X_probe, compression=3
)
integrity_probability = frozen_pipeline.predict_proba(X_probe)[:, 1]
assert np.isfinite(integrity_probability).all()
assert ((integrity_probability >= 0.0) & (integrity_probability <= 1.0)).all()

model_fit_audit = {
    "selected_model": SELECTED_MODEL,
    "selected_trial_number": SELECTED_TRIAL_NUMBER,
    "train_events": int(len(candidate_panel)),
    "train_symbols": int(candidate_panel["symbol"].nunique()),
    "raw_model_features": int(len(MODEL_FEATURES)),
    "transformed_model_features": int(len(transformed_feature_names)),
    "class_weight_mode": class_weight_mode,
    "scale_pos_weight": float(scale_pos_weight),
    "average_uniqueness_weighting_applied": True,
    "sample_weight_mean": float(full_train_sample_weight.mean()),
    "sample_weight_min": float(full_train_sample_weight.min()),
    "sample_weight_max": float(full_train_sample_weight.max()),
    "fit_seconds": float(fit_seconds),
    "training_metrics_reported": False,
    "calibration_fitted": False,
    "threshold_selected": False,
    "probe_probability_min": float(integrity_probability.min()),
    "probe_probability_max": float(integrity_probability.max()),
    "unseen_test_used": False,
    **serialization_audit,
}
model_fit_audit_df = pd.DataFrame([model_fit_audit])
model_fit_audit_df.to_csv(
    RESULT_PATHS["audits"] / "07_model_fit_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(model_fit_audit_df)


## 6. Model card, manifest, and integrity checks

In [ ]:
model_card = {
    "model_name": "frozen_xgboost_primary",
    "stage": 7,
    "status": "fitted_train_only_not_yet_tested",
    "selected_trial_number": SELECTED_TRIAL_NUMBER,
    "hyperparameters": SELECTED_PARAMS,
    "random_seed": SEED,
    "training_population": {
        "events": int(len(candidate_panel)),
        "symbols": int(candidate_panel["symbol"].nunique()),
        "first_event_start_date": str(candidate_panel["dEven"].min()),
        "last_event_start_date": str(candidate_panel["dEven"].max()),
        "last_event_end_date": str(candidate_panel["event_end_date"].max()),
        "positive_fraction": float(candidate_panel["meta_label"].mean()),
        "training_population_sha256": training_population_fingerprint,
    },
    "features": {
        "raw_feature_count": len(MODEL_FEATURES),
        "numeric_feature_count": len(NUMERIC_FEATURES),
        "categorical_features": CATEGORICAL_FEATURES,
        "transformed_feature_count": len(transformed_feature_names),
    },
    "sample_weighting": {**weight_summary, "weights_sha256": weights_fingerprint},
    "serialization": serialization_audit,
    "probability_policy": {
        "calibration_fitted": False,
        "threshold_selected": False,
        "raw_probabilities_may_be_used_for_ranking": True,
        "raw_probabilities_must_not_be_interpreted_as_calibrated": True,
    },
    "training_metrics_reported": False,
    "unseen_test_used": False,
}
model_card_path = RESULT_PATHS["models"] / "07_frozen_xgboost_model_card.json"
model_card_path.write_text(
    json.dumps(model_card, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

manifest = {
    "stage": 7,
    "status": "frozen_after_integrity_checks",
    "stage_version": "v1_full_train_average_uniqueness_xgboost",
    "schema_version": FROZEN_TRAINING_SCHEMA_VERSION,
    "notebook": "07_frozen_model_training.ipynb",
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "software": software_manifest(),
    "lineage": {
        "stage05_configuration_hash": stage05_manifest["configuration_hash"],
        "stage06_code_commit_sha": stage06_manifest["git_commit_sha"],
        "stage06_study_signature": stage06_manifest["study_signature"],
        "stage06_selected_model": SELECTED_MODEL,
        "stage06_selected_trial_number": SELECTED_TRIAL_NUMBER,
    },
    "training_population": model_card["training_population"],
    "features": {
        "model_features": MODEL_FEATURES,
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "transformed_features": transformed_feature_names,
    },
    "sample_weighting": model_card["sample_weighting"],
    "model": {
        "selected_model": SELECTED_MODEL,
        "selected_trial_number": SELECTED_TRIAL_NUMBER,
        "hyperparameters": SELECTED_PARAMS,
        "class_weight_mode": class_weight_mode,
        "scale_pos_weight": float(scale_pos_weight),
        "random_seed": SEED,
        "fit_seconds": float(fit_seconds),
        **serialization_audit,
    },
    "probability_policy": model_card["probability_policy"],
    "training_metrics_reported": False,
    "validation_data_used_for_fit": False,
    "unseen_test_used": False,
    "configuration_hash": stable_object_hash({
        "frozen_training_config": frozen_training_config,
        "stage06_decision": stage06_decision,
        "best_hyperparameters": best_hyperparameters,
        "feature_manifest": feature_manifest_df.to_dict(orient="records"),
        "training_population_sha256": training_population_fingerprint,
        "weights_sha256": weights_fingerprint,
    }),
}
manifest_path = RESULT_PATHS["manifests"] / "07_frozen_model_training_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

assert candidate_errors_df.empty
assert calendar_errors_df.empty
assert len(candidate_panel) == 118_464
assert candidate_panel["symbol"].nunique() == 499
assert len(MODEL_FEATURES) == 35
assert len(NUMERIC_FEATURES) == 34
assert CATEGORICAL_FEATURES == ["gmma_state"]
assert len(full_train_weights_df) == len(candidate_panel)
assert full_train_weights_df["event_id"].is_unique
assert full_train_weight_audit_df["validation_events_used"].eq(0).all()
assert full_train_weight_audit_df["unseen_test_events_used"].eq(0).all()
assert full_train_weight_audit_df["nonfinite_raw_uniqueness"].eq(0).all()
assert full_train_weight_audit_df["nonfinite_sample_weight"].eq(0).all()
assert full_train_weight_audit_df["nonpositive_sample_weight"].eq(0).all()
assert np.isclose(full_train_weight_audit_df["mean_sample_weight"].iloc[0], 1.0)
assert SELECTED_MODEL == "xgboost"
assert SELECTED_TRIAL_NUMBER == 25
assert class_weight_mode == "none"
assert np.isclose(scale_pos_weight, 1.0)
assert model_path.exists() and model_path.stat().st_size > 0
assert serialization_audit["reload_equivalence_passed"] is True
assert serialization_audit["reload_probability_max_abs_difference"] <= 1e-12
assert model_fit_audit_df["training_metrics_reported"].eq(False).all()
assert model_fit_audit_df["calibration_fitted"].eq(False).all()
assert model_fit_audit_df["threshold_selected"].eq(False).all()
assert model_fit_audit_df["unseen_test_used"].eq(False).all()
assert manifest["unseen_test_used"] is False
assert manifest["validation_data_used_for_fit"] is False

print("Notebook 07 integrity checks: PASSED")
print("Selected primary model: XGBoost")
print("Selected Stage 06 trial:", SELECTED_TRIAL_NUMBER)
print("Complete train candidate events:", len(candidate_panel))
print("Train symbols:", candidate_panel["symbol"].nunique())
print("Raw model features:", len(MODEL_FEATURES))
print("Transformed model features:", len(transformed_feature_names))
print("Average Uniqueness scope: complete train, within symbol")
print("Mean normalized sample weight:", float(full_train_sample_weight.mean()))
print("Validation events used for fit: 0")
print("Unseen-test events used for fit: 0")
print("Training discrimination metrics reported: False")
print("Calibration fitted: False")
print("Threshold selected: False")
print("Model reload equivalence: PASSED")
print("Frozen model SHA256:", serialization_audit["model_sha256"])
print("Manifest code SHA:", manifest["git_commit_sha"])
print(
    "Next stage: freeze a train-only probability and signal policy before "
    "confirmatory unseen-test use."
)


## Review before freezing Stage 07

Inspect all `07_` audit and manifest outputs, plus the local model card. Keep the
binary model and event-level weights locally. Do not run Notebook 08 until this
stage is audited and frozen.